# 04 — Feature correlation and selection

**Notebook 4 of 9 — Drosophila book-chapter production pipeline**

**Purpose:** Assess redundancy among 55 usable engineered features and export the accepted 45-feature production matrix.

**Primary inputs:** `data/processed/drosophila_features_scaled.csv`

**Primary outputs:** `data/processed/drosophila_features_reduced.csv`; `figures/feature_correlation_heatmap.png`

**Manuscript role:** Feature-selection methods; defines the common engineered-feature input for PCA, NMF, and feature characterization.

**Project authors:** Tim Rogalsky, Nicolas Malagon, Lia Campbell-Enns, Matthaeus Dyck

**Reproducibility:** Designed for fresh-kernel execution in production order 01→09 using repository-relative paths. It consumes the explicit scaled-feature interface written by Notebook 03 and requires no private, diagnostic, or obsolete provenance artifact.

A threshold of $|r| > 0.9$ identifies strong redundancy, but pruning is interpretation-driven rather than automatic. Correlated features may remain when they capture distinct aspects of cell dynamics.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

INPUT_PATH = Path("data/processed/drosophila_features_scaled.csv")
OUTPUT_PATH = Path("data/processed/drosophila_features_reduced.csv")
FIGURE_PATH = Path("figures/feature_correlation_heatmap.png")
CORRELATION_THRESHOLD = 0.9

df_scaled = pd.read_csv(INPUT_PATH, index_col=0)
duplicate_columns = df_scaled.columns[df_scaled.columns.duplicated()].tolist()
non_finite_count = int((~np.isfinite(df_scaled.to_numpy())).sum())

print(f"Input path: {INPUT_PATH}")
print(f"Input rows: {df_scaled.shape[0]}")
print(f"Input feature columns: {df_scaled.shape[1]}")
print(f"Duplicate columns ({len(duplicate_columns)}): {duplicate_columns}")
print(f"Non-finite values: {non_finite_count}")

assert df_scaled.shape == (108, 55), f"Expected corrected input shape (108, 55), found {df_scaled.shape}."
assert not duplicate_columns, f"Duplicate feature columns found: {duplicate_columns}"
assert non_finite_count == 0, f"Found {non_finite_count} non-finite values."


## Pearson correlation assessment

Pearson correlations are computed from the 55-feature production table. The heatmap summarizes the full input correlation structure, and the compact pair table reports every unique pair above the working threshold.

In [ ]:
def highly_correlated_pairs(correlation_matrix, threshold=CORRELATION_THRESHOLD):
    """Return unique feature pairs above the absolute-correlation threshold."""
    upper_triangle = np.triu(np.ones(correlation_matrix.shape, dtype=bool), k=1)
    pairs = (
        correlation_matrix.where(upper_triangle)
        .stack()
        .rename("correlation")
        .reset_index()
        .rename(columns={"level_0": "feature_1", "level_1": "feature_2"})
    )
    pairs["abs_correlation"] = pairs["correlation"].abs()
    return (
        pairs.loc[pairs["abs_correlation"] > threshold]
        .sort_values("abs_correlation", ascending=False)
        .reset_index(drop=True)
    )

corr_matrix = df_scaled.corr(method="pearson")
high_corr_before = highly_correlated_pairs(corr_matrix)

print(f"Feature pairs with |r| > {CORRELATION_THRESHOLD}: {len(high_corr_before)}")
display(high_corr_before.round(3))


### Correlation heatmap

The heatmap represents the scaled production input before pruning.

In [ ]:
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(15, 13))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Pearson Correlations Among Corrected Engineered Features")
ax.tick_params(axis="x", labelsize=6)
ax.tick_params(axis="y", labelsize=6)
fig.tight_layout()
fig.savefig(FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()
print(f"Figure saved to: {FIGURE_PATH}")


## Selective feature pruning

The approved pruning set removes 10 unique features based on statistical, mathematical, and biological interpretability:

- `a_range` is removed while retaining the standard, directly interpretable variability descriptor `a_std`.
- `b_total_power` is removed because integration of the periodogram power spectral density makes total spectral power effectively mathematically redundant with signal variance and therefore with `a_std`.
- Four aggregate magnitude descriptors are removed where they duplicate general variability.
- Two oscillation-timing descriptors and two directional magnitude descriptors are removed in favor of retained counterparts.

`f_delta_mean` is retained as a distinct early-to-late change descriptor. With observed-span feature engineering, its correlation with `f_late_mean` is below the pruning threshold. Other strong correlations may remain for conceptually distinct properties such as early-window variability, directional oscillation amplitudes, sharpness, and polynomial coefficients; these are documented rather than automatically pruned.

In [ ]:
features_to_drop = [
    # Statistical redundancy
    "a_range",

    # Spectral redundancy
    "b_total_power",

    # Magnitude redundancy
    "d_mean_magnitude",
    "d_std_magnitude",
    "d_mean_peak_to_trough_magnitude",
    "d_mean_trough_to_peak_magnitude",

    # Oscillation timing redundancy
    "c_avg_time_between_troughs",
    "c_oscillation_rate",

    # Directional magnitude redundancy
    "d_initial_trough_to_peak_magnitude",
    "d_final_trough_to_peak_magnitude",
]

assert len(features_to_drop) == len(set(features_to_drop)) == 10, "The approved drop list must contain 10 unique features."
missing_features = sorted(set(features_to_drop) - set(df_scaled.columns))
assert not missing_features, f"Approved pruning features missing from input: {missing_features}"
assert "a_std" not in features_to_drop and "f_delta_mean" not in features_to_drop

df_reduced = df_scaled.drop(columns=features_to_drop)
high_corr_after = highly_correlated_pairs(df_reduced.corr(method="pearson"))

print(f"Features removed ({len(features_to_drop)}):")
for feature in features_to_drop:
    print(f"  - {feature}")
print(f"Retained features: {df_reduced.shape[1]}")
print(f"Remaining feature pairs with |r| > {CORRELATION_THRESHOLD}: {len(high_corr_after)}")
display(high_corr_after.round(3))

assert df_reduced.shape == (108, 45), f"Expected corrected reduced shape (108, 45), found {df_reduced.shape}."


## Export reduced feature table

The 45-feature result follows from the 55-feature input and the approved 10-feature pruning set. Runtime assertions verify the output shape, column order, and finite values before the table is consumed by the PCA, NMF, and feature-characterization notebooks.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_reduced.to_csv(OUTPUT_PATH)

exported = pd.read_csv(OUTPUT_PATH, index_col=0)
assert exported.shape == df_reduced.shape, "Exported table shape does not match the reduced table."
assert exported.columns.tolist() == df_reduced.columns.tolist(), "Exported feature columns do not match."

print(f"Output path: {OUTPUT_PATH}")
print(f"Output shape: {exported.shape}")
